In [ ]:
!pip install -q xgboost shap openpyxl



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, RocCurveDisplay,
                             classification_report)
from sklearn.inspection import permutation_importance

import xgboost as xgb
import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
print('All libraries loaded.')


All libraries loaded.


In [ ]:
# --- Upload the 5 raw files (Colab file picker) ---
# In Google Colab, run this cell and select the 5 xlsx files when prompted:
# 4_years_datas.xlsx, 5_years_datas.xlsx, 6_years_datas.xlsx, 7_years_datas.xlsx, 8_years_datas.xlsx
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print('Not running in Colab — place the 5 xlsx files in the working directory instead.')


In [ ]:
FILES = {
    4: '4 years_datas.xlsx',
    5: '5 years_datas.xlsx',
    6: '6 years_datas.xlsx',
    7: '7 years_datas.xlsx',
    8: '8 years_datas.xlsx',
}

frames = []
for years, path in FILES.items():
    d = pd.read_excel(path)
    d['followup_years'] = years
    frames.append(d)
raw = pd.concat(frames, ignore_index=True)

print('Combined raw shape:', raw.shape)
print('Total unique patients:', raw['patientid'].nunique())
raw.head()


In [ ]:
# Sanity-check our understanding of type_1 / type_2 / type_3 / type_1_2
chk = raw.dropna(subset=['type_1', 'type_2', 'type_3', 'type_1_2']).copy()
print('type_3 == max(type_1,type_2) match rate:', (chk['type_3'] == chk[['type_1','type_2']].max(axis=1)).mean())
print('type_1_2 == type_1+type_2 match rate   :', (chk['type_1_2'] == chk[['type_1','type_2']].sum(axis=1)).mean())


In [ ]:
baseline = raw[raw['timed'] == 1].copy()
followup = raw[raw['timed'] == 2][['patientid', 'type_3']].rename(columns={'type_3': 'type_3_followup'})
merged = baseline.merge(followup, on='patientid', how='left')

cohort = merged[merged['type_3'] == 1].copy()               # prediabetic at baseline
cohort = cohort.dropna(subset=['type_3_followup']).copy()   # drop lost-to-follow-up
cohort['progressed_to_t2d'] = (cohort['type_3_followup'] == 2).astype(int)

# feature engineering (baseline-only)
for c in ['age', 'weight', 'height', 'glu', 'HbA1c', 'rate', 'SBP', 'DBP']:
    cohort[c] = pd.to_numeric(cohort[c], errors='coerce')
cohort['bmi'] = cohort['weight'] / (cohort['height'] / 100) ** 2
cohort['pulse_pressure'] = cohort['SBP'] - cohort['DBP']
cohort['sex'] = pd.to_numeric(cohort['sex'], errors='coerce').astype('Int64')

FEATURE_COLS = ['age', 'sex', 'bmi', 'glu', 'HbA1c', 'SBP', 'DBP', 'pulse_pressure', 'rate']
df = cohort[['patientid', 'followup_years'] + FEATURE_COLS + ['progressed_to_t2d']].reset_index(drop=True)

print('Cohort shape (baseline-prediabetic, known outcome):', df.shape)
print(df['progressed_to_t2d'].value_counts())
print((df['progressed_to_t2d'].value_counts(normalize=True) * 100).round(1))
df.head()


In [ ]:
df[FEATURE_COLS + ['progressed_to_t2d']].describe().T


In [ ]:
print('Missing values (%):')
print((df[FEATURE_COLS].isna().mean() * 100).round(1).sort_values(ascending=False))


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

counts = df['progressed_to_t2d'].value_counts().sort_index()

ax.bar(['Did not progress', 'Progressed to T2D'],
       counts.values,
       color=['#4C72B0', '#C44E52'])

for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v}\n({v/counts.values.sum()*100:.1f}%)', ha='center')

ax.set_title('Class distribution: progression to T2D at follow-up', pad=25)

ax.set_ylabel('Number of patients')

plt.tight_layout()
plt.show()


In [ ]:
num_feats = ['age', 'bmi', 'glu', 'HbA1c', 'SBP', 'DBP', 'rate']
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
axes = axes.flatten()
for i, col in enumerate(num_feats):
    sns.kdeplot(data=df, x=col, hue='progressed_to_t2d', fill=True, common_norm=False,
                alpha=0.4, ax=axes[i], palette=['#4C72B0', '#C44E52'], legend=(i == 0))
    axes[i].set_title(col)
for j in range(len(num_feats), len(axes)):
    axes[j].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
corr = df[num_feats + ['sex', 'progressed_to_t2d']].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, square=True)
ax.set_title('Correlation matrix (baseline features + outcome)')
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(num_feats), figsize=(18, 4))
for i, col in enumerate(num_feats):
    sns.boxplot(y=df[col], ax=axes[i], color='#55A868')
    axes[i].set_title(col)
plt.tight_layout(); plt.show()
print('Extreme glucose/HbA1c values are physiologically plausible for patients near the diabetes',
      'threshold and are kept; they are exactly the borderline cases the model needs to learn from.')


In [ ]:
print('Progression rate (%) by sex (0=female,1=male):')
print((pd.crosstab(df['sex'], df['progressed_to_t2d'], normalize='index') * 100).round(1))
print()
print('Progression rate (%) by follow-up duration (years):')
print((pd.crosstab(df['followup_years'], df['progressed_to_t2d'], normalize='index') * 100).round(1))


In [ ]:
X = df[FEATURE_COLS].copy()
y = df['progressed_to_t2d'].copy()

# 70 / 15 / 15 stratified split (train / val / test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE)
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

# Train+Val is used for 5-fold CV (robust estimate, small-N problem); held-out Test (15%) never touched until final evaluation
X_trainval = pd.concat([X_train, X_val]); y_trainval = pd.concat([y_train, y_val])

preprocess = ColumnTransformer([
    ('impute_scale', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), FEATURE_COLS),
])


In [ ]:
neg, pos = np.bincount(y_trainval)
scale_pos_weight = neg / pos

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=400, max_depth=6, min_samples_leaf=5,
                                             class_weight='balanced', random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                                     random_state=RANDOM_STATE),
    'XGBoost': xgb.XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                  scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                                  random_state=RANDOM_STATE),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, gamma='scale', probability=True,
                      class_weight='balanced', random_state=RANDOM_STATE),
}
print(f'Class imbalance ratio (neg/pos) used for weighting: {scale_pos_weight:.2f}')


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_rows, fitted_pipelines, test_rows, roc_data = [], {}, [], {}

for name, clf in models.items():
    pipe = Pipeline([('prep', preprocess), ('clf', clf)])
    cvres = cross_validate(pipe, X_trainval, y_trainval, cv=cv, scoring=scoring)
    cv_rows.append({
        'Model': name,
        'CV Accuracy': cvres['test_accuracy'].mean(),
        'CV Precision': cvres['test_precision'].mean(),
        'CV Recall': cvres['test_recall'].mean(),
        'CV F1': cvres['test_f1'].mean(),
        'CV ROC-AUC': cvres['test_roc_auc'].mean(),
        'CV ROC-AUC (std)': cvres['test_roc_auc'].std(),
    })
    # fit on the full train+val set, then evaluate once on the untouched test set
    pipe.fit(X_trainval, y_trainval)
    fitted_pipelines[name] = pipe
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    test_rows.append({
        'Model': name,
        'Test Accuracy': accuracy_score(y_test, y_pred),
        'Test Precision': precision_score(y_test, y_pred),
        'Test Recall': recall_score(y_test, y_pred),
        'Test F1': f1_score(y_test, y_pred),
        'Test ROC-AUC': roc_auc_score(y_test, y_proba),
    })
    roc_data[name] = (y_test, y_proba)

cv_df = pd.DataFrame(cv_rows).sort_values('CV ROC-AUC', ascending=False)
test_df = pd.DataFrame(test_rows).sort_values('Test ROC-AUC', ascending=False)
print('===== 5-fold CV comparison (train+val) ====='); display(cv_df.round(3))
print('===== Held-out TEST set comparison ====='); display(test_df.round(3))


In [ ]:
for name, clf in models.items():
    pipe = fitted_pipelines[name]
    y_pred = pipe.predict(X_test)
    print(f'=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=['No T2D', 'T2D']))
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No T2D', 'T2D'], yticklabels=['No T2D', 'T2D'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(f'Confusion Matrix — {name}')
    plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, (yt, yp) in roc_data.items():
    RocCurveDisplay.from_predictions(yt, yp, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle='--', color='gray')
ax.set_title('ROC curves — held-out test set')
plt.tight_layout(); plt.show()


In [ ]:
melted = test_df.melt(id_vars='Model', value_vars=['Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1', 'Test ROC-AUC'])
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=melted, x='variable', y='value', hue='Model', ax=ax)
ax.set_ylim(0, 1); ax.set_ylabel('Score'); ax.set_xlabel('')
ax.set_title('Model comparison — held-out test set')
plt.xticks(rotation=15); plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()


In [ ]:
# Pick the best tree-based model (fast, exact SHAP via TreeExplainer)
tree_candidates = [m for m in ['XGBoost', 'Gradient Boosting', 'Random Forest'] if m in fitted_pipelines]
best_tree_name = max(tree_candidates, key=lambda m: test_df.set_index('Model').loc[m, 'Test ROC-AUC'])
best_pipe = fitted_pipelines[best_tree_name]
print('Explaining model:', best_tree_name)

# transform test features exactly as the pipeline would, so SHAP sees what the model sees
X_test_transformed = best_pipe.named_steps['prep'].transform(X_test)
X_test_df = pd.DataFrame(X_test_transformed, columns=FEATURE_COLS, index=X_test.index)

explainer = shap.TreeExplainer(best_pipe.named_steps['clf'])
shap_values = explainer.shap_values(X_test_df)


In [ ]:
shap.summary_plot(shap_values, X_test_df, show=False)
plt.tight_layout(); plt.show()


In [ ]:
shap.summary_plot(shap_values, X_test_df, plot_type='bar', show=False)
plt.tight_layout(); plt.show()


In [ ]:
explainer = shap.TreeExplainer(best_pipe.named_steps['clf'])
raw_shap = explainer.shap_values(X_test_df)

if isinstance(raw_shap, list):
    shap_values = raw_shap[1]
    base_value = explainer.expected_value[1] if hasattr(explainer.expected_value, '__len__') else explainer.expected_value
elif np.ndim(raw_shap) == 3:
    shap_values = raw_shap[:, :, 1]
    ev = explainer.expected_value
    base_value = ev[1] if hasattr(ev, '__len__') else ev
else:
    shap_values = raw_shap
    ev = explainer.expected_value
    base_value = ev[0] if hasattr(ev, '__len__') else ev

In [ ]:
# Local explanation: why did the model flag/clear one specific patient?
i = 0  # change this index to inspect a different test-set patient
shap.force_plot(base_value, shap_values[i], X_test_df.iloc[i], matplotlib=True, show=False)
plt.tight_layout(); plt.show()
print('Actual outcome for this patient:', 'Progressed to T2D' if y_test.iloc[i] == 1 else 'Did not progress')


In [ ]:
result = permutation_importance(best_pipe, X_test, y_test, n_repeats=30,
                                 random_state=RANDOM_STATE, scoring='roc_auc')
imp_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std,
}).sort_values('importance_mean', ascending=False)
display(imp_df.round(4))

fig, ax = plt.subplots(figsize=(6, 5))
order = imp_df.sort_values('importance_mean')
ax.barh(order['feature'], order['importance_mean'], xerr=order['importance_std'], color='#4C72B0')
ax.set_xlabel('Permutation importance (drop in ROC-AUC when shuffled)')
ax.set_title(f'Feature importance — {best_tree_name}')
plt.tight_layout(); plt.show()


In [ ]:
def risk_band(p):
    if p < 0.33: return 'Low'
    elif p < 0.66: return 'Moderate'
    else: return 'High'

best_overall = test_df.iloc[0]['Model']
proba_best = fitted_pipelines[best_overall].predict_proba(X_test)[:, 1]
risk_table = X_test.copy()
risk_table['true_outcome'] = y_test.values
risk_table['predicted_probability'] = proba_best.round(3)
risk_table['risk_band'] = [risk_band(p) for p in proba_best]
print(f'Best overall model by test ROC-AUC: {best_overall}')
risk_table.sample(10, random_state=1)
